# 2. SQL para exploración de datos (Data Discovery)
____
Se debe importar el dataset usando pandas, se escriben las consultas en el archivo `02_sql_discovery.sql` y se cargan a este notebook de jupyter como una lista de queries.

In [ ]:
import sqlite3
import pandas as pd
import csv

ventas_techventas = "../data/ventas_techventas.csv"
codificacion = "latin-1"

dataFrame = pd.read_csv(ventas_techventas, encoding=codificacion, quoting=csv.QUOTE_NONE)

# El el dataset, el nombre de Ana López tiene un error de codificación, por lo que se reemplaza al cargarlo
# De igual manera, se eliminan las comillas que hay en algunos de los registros y en algunos de los productos
dataFrame["vendedor"] = dataFrame["vendedor"].str.replace("Ana Lï¿½ï¿", "Ana López")
dataFrame['order_id'] = dataFrame['order_id'].str.strip('"')
dataFrame['vendedor'] = dataFrame['vendedor'].str.strip('"')
dataFrame['producto'] = dataFrame['producto'].str.replace('""', '"')

# Creación de la base de datos sqlite en la memoria
connection = sqlite3.connect(":memory:")

filas_insertadas = dataFrame.to_sql("ventas_techventas", connection, if_exists="replace", index=False)
print(f"Filas insertadas desde el dataset a la base de datos sqlite: {filas_insertadas}")

#### **Aclaración del ejercicio**
Al igual que en la parte 1 del reto, dado que la columna de cantidad tiene valores negativos, algunos de los resultados de las queries se verán afectadas por este dato, lo que indica que en la fase siguiente con pandas es necesario hacer un filtro de ese dato negativo para evitar resultados alterados.

### Importación del archivo .sql
____

In [ ]:
def cargar_queries_sql(ruta_de_archivo: str) -> list:
    with open(ruta_de_archivo) as archivo:
        contenido = archivo.read()
        queries = []

        # Se separa el archivo por queries a partir del caracter ;
        for query in contenido.split(";"):
            if query.strip():
                queries.append(query)

        return queries

In [ ]:
queries_sql = "../sql/02_sql_discovery.sql"
queries = cargar_queries_sql(queries_sql)

### Query 1: SELECT DISTINCT
____
Se deben listar todos los productos únicos disponibles con un alias para la columna

In [ ]:
productos_unicos = pd.read_sql_query(queries[0], connection)
# Entre los valores que se imprimen también aparece el valor nulo que hay en uno de los pedidos
display(productos_unicos)

### Query 2: WHERE + BETWEEN
----
Se filtran los pedidos del primer trimestre (enero a mar 2024) cuya cantidad sea mayor a 5 unidades

In [ ]:
pedidos_filtrados = pd.read_sql_query(queries[1], connection)
display(pedidos_filtrados)

### Query 3: GROUP BY + HAVING
----
Se muestran los vendedores cuyo ingreso bruto total supera los 10 mil dólares

In [ ]:
vendedores = pd.read_sql_query(queries[2], connection)
# Se debe tener en cuenta la venta con cantidad negativa que tuvo la vendedora Ana, lo que altera su ingreso bruto total
display(vendedores)

### Query 4: COUNT + SUM + AVG
----
Se muestra el total de pedidos, suma de unidades y precio unitario promedio de cada categoría

In [ ]:
filtro_categorias = pd.read_sql_query(queries[3], connection)
display(filtro_categorias)

### QUERY 5: JOIN
----
Crear una tabla para los vendedores, y calcular posteriormente el porcentaje de cumplimiento de la meta mensual de cada uno

In [ ]:
query_tabla_vendedores = """--sql
    CREATE TABLE "vendedores" (
        vendedor TEXT,
        zona TEXT,
        meta_mensual INT
    );
"""

query_insertar_datos = """--sql
    INSERT INTO "vendedores" (vendedor, zona, meta_mensual) VALUES 
    ('Ana López', 'Norte', 8000),
    ('Carlos Ruiz', 'Sur', 7500),
    ('Luis Mora', 'Norte', 8000),
    ('Maria Torres', 'Centro', 7000);
"""

cursor = connection.cursor()

cursor.execute(query_tabla_vendedores)
cursor.execute(query_insertar_datos)
connection.commit()

**Debido a esta tabla, fue necesario hacer el reemplazo del nombre de Ana López que tenía el problema de codificación al inicio**

#### Cálculo del porcentaje de cumplimiento por mes

In [ ]:
cumplimiento_vendedores = pd.read_sql_query(queries[4], connection)
display(cumplimiento_vendedores)

### Query 6: SUBCONSULTA
----
Encontrar el cliente que generó el ingreso total en el primer semestre

In [ ]:
cliente_mayor = pd.read_sql_query(queries[5], connection)
display(cliente_mayor)

In [ ]:
connection.close()